# Exploratory Data Analysis — Xente Transaction Data

Exploration only — no production feature engineering here (that lives in
`src/data_processing.py`). Goal: understand structure, quality, and shape
of the raw data to inform feature engineering and the RFM proxy target.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/data.csv")
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])
df.shape

## 1. Overview of the Data

In [ ]:
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
df.dtypes

In [ ]:
df.head()

## 2. Summary Statistics

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().T

## 3. Distribution of Numerical Features

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))
if len(numeric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], bins=40, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Distribution of Categorical Features

In [ ]:
categorical_cols = [
    c for c in ["ProductCategory", "ChannelId", "ProviderId", "PricingStrategy", "CurrencyCode"]
    if c in df.columns
]
fig, axes = plt.subplots(1, len(categorical_cols), figsize=(5 * len(categorical_cols), 4))
if len(categorical_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, categorical_cols):
    df[col].astype(str).value_counts().head(10).plot(kind="bar", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix (Numerical Features)")
plt.show()

## 6. Missing Values

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0")

## 7. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))
if len(numeric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Key Insights

_Fill in after running against the real Xente dataset. Example prompts to answer:_

1. **Amount distribution**: Is `Amount` heavily skewed / does it include negative values (credits)? What does that imply for aggregate features like `total_amount`?
2. **Transaction volume per customer**: Is activity concentrated in a small number of customers, or fairly even? This affects how meaningful `transaction_count` is as a risk signal.
3. **Categorical concentration**: Are one or two `ProductCategory` / `ChannelId` values dominant? Consider whether rare categories need grouping before one-hot encoding.
4. **Missing values**: Which columns need imputation vs. removal, and why?
5. **Outliers**: Are extreme `Amount` values genuine high-value customers or data entry errors — and should they be capped before RFM clustering (K-Means is sensitive to outliers)?
